In [ ]:
from itertools import combinations
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
import pandas as pd

In [ ]:
df = pd.read_json('book_engagement_metrics.json', lines=True)
df

,id,like_count,region_code,video_description,favorites_count,hashtag_names,effect_ids,is_stem_verified,playlist_id,username,...,video_tag,full_text,book_title,view_count_pct_rank,like_count_pct_rank,share_count_pct_rank,favorites_count_pct_rank,comment_count_pct_rank,raw_engagement_metrics,engagement_metrics
0,7455100765804285206,331,PL,#FLAWLESS | powtarzam pożar pożar || #booktok ...,55,"[flawless, beya, allofyourflaws, booktok, book...",[],False,0,kulka_czytka,...,None,#flawless | powtarzam pożar pożar || #booktok ...,Other / None,0.818403,0.885508,0.907470,0.896304,0.707109,4.214794,84.295885
1,7455099290818563350,12,PL,"люблю цю історію😭 я ще не дочитала, але ця кни...",0,"[reading, книги, читання, booktok, ninthhouse,...",[],False,0,i_am_.who,...,None,"люблю цю історію😭 я ще не дочитала, але ця кни...",Other / None,0.061381,0.098196,0.175980,0.051473,0.163322,0.550351,11.007022
2,7455097266605427990,269,PL,Kochajmy Alexa #viral #booktok #ksiazkara #ann...,43,"[dc, viral, booktok, annahuang, ksiazkara]",[18702636],False,0,zaczytana.duszyczka,...,None,kochajmy alexa #viral #booktok #ksiazkara #ann...,Other / None,0.883012,0.870197,0.875375,0.877063,0.755968,4.261615,85.232300
3,7455096763297303830,1878,PL,Najlepszy i najgroźniejszy 🤫 #seriahellish #he...,464,"[dc, aidenscott, dlaciebie, nicholasscott, hid...",[],False,0,yrdarkness,...,None,najlepszy i najgroźniejszy 🤫 #seriahellish #he...,"Summary: I Wanna Fall, Seria Hellish, Seria Agony",0.964931,0.973103,0.981315,0.981003,0.976048,4.876399,97.527989
4,7455095111274237206,63,PL,Boże jak mi się podoba ten tik tok😍 nie widzia...,11,"[dc, books, shey, fyp, dlaciebie, pageforyou, ...",[],False,0,bryszkaczyta,...,None,boże jak mi się podoba ten tik tok😍 nie widzia...,Trylogia Hell,0.278399,0.536678,0.585976,0.668978,0.538853,2.608884,52.177687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51265,7590068290337344791,23,pl,Wygrał.. #funny #book #booktok #państwamiasta ...,4,"[funny, book, państwamiasta, booktok, zaksiazk...",[],False,0,busia.fantasy_booktok,...,None,wygrał.. #funny #book #booktok #państwamiasta ...,Other / None,0.353881,0.206700,0.671338,0.395797,0.400575,2.028291,40.565828
51266,7590067926821293315,1016,pl,#booktokpolska #booktok #wattpadbook #wattpadp...,386,"[wattpadbook, romancebook, booktok, wattpadpol...",[],False,0,thecureforsoul,...,None,#booktokpolska #booktok #wattpadbook #wattpadp...,Other / None,0.956661,0.953335,0.924469,0.977258,0.538853,4.350575,87.011508
51267,7590067535052033302,6,pl,Happy New Year!!! #heatedrivalry #booktok #sha...,2,"[thelonggame, booktok, heatedrivalry, ilyaroza...",[],False,0,bookish.a2,...,None,happy new year!!! #heatedrivalry #booktok #sha...,Other / None,0.018149,0.041681,0.585976,0.245407,0.163322,1.054535,21.090696
51268,7590067529234517271,119,pl,#breakermk 💚 | @martyna keller #booktok #dlaci...,31,"[viral, fyp, dlaciebie, booktok, breakermk]",[],False,0,booksby.nala,...,None,#breakermk 💚 | @martyna keller #booktok #dlaci...,Other / None,0.747055,0.733694,0.442900,0.846801,0.400575,3.171026,63.420519


In [ ]:
df.columns

Index(['id', 'like_count', 'region_code', 'video_description',
       'favorites_count', 'hashtag_names', 'effect_ids', 'is_stem_verified',
       'playlist_id', 'username', 'video_duration', 'video_label',
       'comment_count', 'create_time', 'music_id', 'view_count',
       'video_mention_list', 'voice_to_text', 'hashtag_info_list',
       'share_count', 'sticker_info_list', 'effect_info_list', 'video_tag',
       'full_text', 'book_title', 'view_count_pct_rank', 'like_count_pct_rank',
       'share_count_pct_rank', 'favorites_count_pct_rank',
       'comment_count_pct_rank', 'raw_engagement_metrics',
       'engagement_metrics'],
      dtype='object')

# Preparing files for Network Analysis

In [ ]:
G_creator_book = nx.Graph()

In [ ]:
for _, row in df.iterrows():
    creator = str(row['username']).strip()
    books = row['book_title']

    if creator and books:
        for book in books:
            if G_creator_book.has_edge(creator, book):
                G_creator_book[creator][book]['weight'] += 1
            else:
                G_creator_book.add_node(creator, node_type='creator')
                G_creator_book.add_node(book, node_type='book')
                G_creator_book.add_edge(
                    creator, book, weight=1, relation='created'
                )

In [ ]:
#export the graph to a GEXF file
nx.write_gexf(G_creator_book, 'network_creator_book.gexf')

In [ ]:
G_book_cooc = nx.Graph()

In [ ]:
for books in df['book_title']:
    if len(books) > 1:
        for b1, b2 in combinations(sorted(books), 2):
            if G_book_cooc.has_edge(b1, b2):
                G_book_cooc[b1][b2]['weight'] += 1
            else:
                G_book_cooc.add_edge(b1, b2, weight=1)

In [ ]:
nx.write_gexf(G_book_cooc, 'network_book_cooccurrence.gexf')

In [ ]:
creator_nodes = [
    n
    for n, d in G_creator_book.nodes(data=True)
    if d.get('node_type') == 'creator'
]

In [ ]:
book_nodes = [
    n
    for n, d in G_creator_book.nodes(data=True)
    if d.get('node_type') == 'book'
]

In [ ]:
df_creators_top = pd.DataFrame({
    'Creator': creator_nodes,
    'Number of Unique Books': [
        G_creator_book.degree(n) for n in creator_nodes
    ],
    'Total Number of Movies': [
        G_creator_book.degree(n, weight='weight') for n in creator_nodes
    ],
}).sort_values(by='Total Number of Movies', ascending=False)

In [ ]:
print(df_creators_top.head(10).to_string(index=False))